# SQL Analysis in Google Colab using Natural Language Prompts

This notebook demonstrates how to:
1. Generate sample data
2. Set up SQLite database in Colab
3. Use natural language prompts to query data (using Claude API)
4. Perform SQL analysis

In [4]:
!pip install langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.3/66.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.6/426.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 489.1/489.1 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.7/233.7 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.43.0
    Uninstalling google-auth-2.43.0:
      Successfully uninstalled google-auth-2.43.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.1
    Uninstalling langchain-core-1.2.1:
      Successfully uninstalled langchain-core-1.2.1
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.55.0
    Uninstalling google-genai-1.55.0:
      Successfully uninstalled google-genai-1.55.0
ERROR: pip's dependency resolver does not currently take into

In [8]:
from langchain_google_genai import GoogleGenerativeAI

In [9]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta
import random
import json



We'll create a sales database with multiple tables:
- **customers**: Customer information
- **products**: Product catalog
- **orders**: Order transactions
- **order_items**: Individual items in each order

In [10]:
# Generate sample customers data
def generate_customers(n=100):
    first_names = ['John', 'Emma', 'Michael', 'Sophia', 'William', 'Olivia', 'James', 'Ava', 'Robert', 'Isabella']
    last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Rodriguez', 'Martinez']
    cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 'Philadelphia', 'San Antonio', 'San Diego', 'Dallas', 'San Jose']

    customers = []
    for i in range(1, n+1):
        customers.append({
            'customer_id': i,
            'first_name': random.choice(first_names),
            'last_name': random.choice(last_names),
            'email': f'customer{i}@email.com',
            'city': random.choice(cities),
            'signup_date': (datetime.now() - timedelta(days=random.randint(1, 730))).strftime('%Y-%m-%d')
        })
    return pd.DataFrame(customers)

# Generate sample products data
def generate_products(n=50):
    categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Books']
    products = []
    for i in range(1, n+1):
        category = random.choice(categories)
        products.append({
            'product_id': i,
            'product_name': f'{category} Item {i}',
            'category': category,
            'price': round(random.uniform(10, 500), 2),
            'stock_quantity': random.randint(0, 200)
        })
    return pd.DataFrame(products)

# Generate sample orders data
def generate_orders(customers_df, n=500):
    orders = []
    for i in range(1, n+1):
        orders.append({
            'order_id': i,
            'customer_id': random.choice(customers_df['customer_id'].tolist()),
            'order_date': (datetime.now() - timedelta(days=random.randint(1, 365))).strftime('%Y-%m-%d'),
            'status': random.choice(['Completed', 'Pending', 'Shipped', 'Cancelled']),
            'total_amount': 0  # Will be calculated later
        })
    return pd.DataFrame(orders)

# Generate sample order items data
def generate_order_items(orders_df, products_df, avg_items=3):
    order_items = []
    item_id = 1
    for order_id in orders_df['order_id']:
        num_items = random.randint(1, avg_items * 2)
        selected_products = random.sample(products_df['product_id'].tolist(), min(num_items, len(products_df)))

        for product_id in selected_products:
            quantity = random.randint(1, 5)
            price = products_df[products_df['product_id'] == product_id]['price'].values[0]
            order_items.append({
                'item_id': item_id,
                'order_id': order_id,
                'product_id': product_id,
                'quantity': quantity,
                'price': price,
                'subtotal': round(price * quantity, 2)
            })
            item_id += 1
    return pd.DataFrame(order_items)

# Generate all data
print("Generating sample data...")
customers_df = generate_customers(100)
products_df = generate_products(50)
orders_df = generate_orders(customers_df, 500)
order_items_df = generate_order_items(orders_df, products_df)

# Update total_amount in orders
order_totals = order_items_df.groupby('order_id')['subtotal'].sum().reset_index()
orders_df = orders_df.merge(order_totals, on='order_id', how='left')
orders_df['total_amount'] = orders_df['subtotal'].fillna(0)
orders_df = orders_df.drop('subtotal', axis=1)

print("Sample data generated successfully!")
print(f"\nCustomers: {len(customers_df)} records")
print(f"Products: {len(products_df)} records")
print(f"Orders: {len(orders_df)} records")
print(f"Order Items: {len(order_items_df)} records")

Generating sample data...
Sample data generated successfully!

Customers: 100 records
Products: 50 records
Orders: 500 records
Order Items: 1750 records


In [11]:
print("\n=== CUSTOMERS TABLE ===")
display(customers_df.head())

print("\n=== PRODUCTS TABLE ===")
display(products_df.head())

print("\n=== ORDERS TABLE ===")
display(orders_df.head())

print("\n=== ORDER_ITEMS TABLE ===")
display(order_items_df.head())


=== CUSTOMERS TABLE ===


,customer_id,first_name,last_name,email,city,signup_date
0,1,Ava,Johnson,customer1@email.com,New York,2024-03-29
1,2,John,Brown,customer2@email.com,Los Angeles,2024-11-27
2,3,James,Brown,customer3@email.com,Houston,2025-09-20
3,4,Emma,Davis,customer4@email.com,San Jose,2024-11-21
4,5,Sophia,Johnson,customer5@email.com,Houston,2025-08-22



=== PRODUCTS TABLE ===


,product_id,product_name,category,price,stock_quantity
0,1,Books Item 1,Books,347.75,61
1,2,Home & Garden Item 2,Home & Garden,166.93,160
2,3,Home & Garden Item 3,Home & Garden,348.79,110
3,4,Home & Garden Item 4,Home & Garden,451.18,137
4,5,Clothing Item 5,Clothing,355.38,101



=== ORDERS TABLE ===


,order_id,customer_id,order_date,status,total_amount
0,1,72,2025-04-08,Shipped,5679.79
1,2,49,2025-05-07,Shipped,2446.22
2,3,34,2025-07-01,Completed,3941.68
3,4,72,2025-08-03,Completed,3523.78
4,5,95,2025-06-08,Cancelled,2168.09



=== ORDER_ITEMS TABLE ===


,item_id,order_id,product_id,quantity,price,subtotal
0,1,1,5,2,355.38,710.76
1,2,1,35,5,249.43,1247.15
2,3,1,31,5,342.58,1712.90
3,4,1,28,3,218.48,655.44
4,5,1,4,3,451.18,1353.54


In [12]:
# Create SQLite database
conn = sqlite3.connect('sales_database.db')
cursor = conn.cursor()

# Load DataFrames into SQLite tables
customers_df.to_sql('customers', conn, if_exists='replace', index=False)
products_df.to_sql('products', conn, if_exists='replace', index=False)
orders_df.to_sql('orders', conn, if_exists='replace', index=False)
order_items_df.to_sql('order_items', conn, if_exists='replace', index=False)

print("Database created and data loaded successfully!")

# Display table schemas
tables = ['customers', 'products', 'orders', 'order_items']
for table in tables:
    print(f"\n=== {table.upper()} TABLE SCHEMA ===")
    cursor.execute(f"PRAGMA table_info({table})")
    schema = cursor.fetchall()
    for col in schema:
        print(f"  {col[1]} ({col[2]})")

Database created and data loaded successfully!

=== CUSTOMERS TABLE SCHEMA ===
  customer_id (INTEGER)
  first_name (TEXT)
  last_name (TEXT)
  email (TEXT)
  city (TEXT)
  signup_date (TEXT)

=== PRODUCTS TABLE SCHEMA ===
  product_id (INTEGER)
  product_name (TEXT)
  category (TEXT)
  price (REAL)
  stock_quantity (INTEGER)

=== ORDERS TABLE SCHEMA ===
  order_id (INTEGER)
  customer_id (INTEGER)
  order_date (TEXT)
  status (TEXT)
  total_amount (REAL)

=== ORDER_ITEMS TABLE SCHEMA ===
  item_id (INTEGER)
  order_id (INTEGER)
  product_id (INTEGER)
  quantity (INTEGER)
  price (REAL)
  subtotal (REAL)


In [29]:
# Install required library first (run this in a cell)
# !pip install google-generativeai -q

import google.generativeai as genai
import pandas as pd
import sqlite3

# Configure the Google API
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the model
llm = genai.GenerativeModel('gemini-3-flash-preview')

# Get database schema for context
def get_database_schema():
    schema_info = """
    Database Schema:

    1. customers table:
       - customer_id (INTEGER): Unique customer identifier
       - first_name (TEXT): Customer's first name
       - last_name (TEXT): Customer's last name
       - email (TEXT): Customer's email address
       - city (TEXT): Customer's city
       - signup_date (TEXT): Date when customer signed up

    2. products table:
       - product_id (INTEGER): Unique product identifier
       - product_name (TEXT): Product name
       - category (TEXT): Product category
       - price (REAL): Product price
       - stock_quantity (INTEGER): Available stock quantity

    3. orders table:
       - order_id (INTEGER): Unique order identifier
       - customer_id (INTEGER): Foreign key to customers table
       - order_date (TEXT): Date when order was placed
       - status (TEXT): Order status (Completed, Pending, Shipped, Cancelled)
       - total_amount (REAL): Total order amount

    4. order_items table:
       - item_id (INTEGER): Unique item identifier
       - order_id (INTEGER): Foreign key to orders table
       - product_id (INTEGER): Foreign key to products table
       - quantity (INTEGER): Quantity of product ordered
       - price (REAL): Price per unit at time of order
       - subtotal (REAL): Total price for this item (quantity * price)
    """
    return schema_info

def natural_language_to_sql(prompt):
    """
    Convert natural language prompt to SQL query using Google Gemini API
    """
    schema = get_database_schema()

    full_prompt = f"""{schema}

Based on the above database schema, convert the following natural language query into a SQL query.
Return ONLY the SQL query without any explanation or markdown formatting.

Natural language query: {prompt}

SQL query:"""

    # Use Gemini's generate_content method
    response = llm.generate_content(full_prompt)

    sql_query = response.text.strip()
    # Remove markdown code blocks if present
    sql_query = sql_query.replace('```sql', '').replace('```', '').strip()

    return sql_query

def execute_natural_language_query(prompt, show_sql=True):
    """
    Execute a natural language query and return results
    """
    print(f"\n🔍 Natural Language Query: {prompt}")
    print("\n⏳ Converting to SQL...")

    sql_query = natural_language_to_sql(prompt)

    if show_sql:
        print(f"\n📝 Generated SQL Query:\n{sql_query}")

    print("\n✅ Executing query...\n")

    try:
        result_df = pd.read_sql_query(sql_query, conn)
        print(f"Results ({len(result_df)} rows):")
        display(result_df)
        return result_df
    except Exception as e:
        print(f"❌ Error executing query: {str(e)}")
        return None

print("Natural Language to SQL converter is ready!")

Natural Language to SQL converter is ready!




Now you can analyze the data using simple English prompts!

### Example 1: Basic Query

In [20]:
# Find top 10 customers by total spending
execute_natural_language_query(
    "Show me the top 10 customers who have spent the most money, including their names and total spending"
)


🔍 Natural Language Query: Show me the top 10 customers who have spent the most money, including their names and total spending

⏳ Converting to SQL...

📝 Generated SQL Query:
SELECT first_name, last_name, SUM(total_amount) AS total_spent
FROM customers
JOIN orders ON customers.customer_id = orders.customer_id
GROUP BY customers.customer_id, first_name, last_name
ORDER BY total_spent DESC
LIMIT 10;

✅ Executing query...

Results (10 rows):


,first_name,last_name,total_spent
0,Sophia,Brown,33012.94
1,Sophia,Williams,31626.49
2,William,Martinez,30425.41
3,Ava,Smith,29713.10
4,William,Davis,27817.22
5,Robert,Davis,27606.20
6,James,Brown,27141.95
7,William,Johnson,27075.01
8,Isabella,Davis,27009.34
9,Robert,Williams,26935.22


,first_name,last_name,total_spent
0,Sophia,Brown,33012.94
1,Sophia,Williams,31626.49
2,William,Martinez,30425.41
3,Ava,Smith,29713.10
4,William,Davis,27817.22
5,Robert,Davis,27606.20
6,James,Brown,27141.95
7,William,Johnson,27075.01
8,Isabella,Davis,27009.34
9,Robert,Williams,26935.22


### Example 2: Sales Analysis

In [21]:
# Analyze sales by category
execute_natural_language_query(
    "What is the total revenue for each product category? Sort by revenue in descending order"
)


🔍 Natural Language Query: What is the total revenue for each product category? Sort by revenue in descending order

⏳ Converting to SQL...

📝 Generated SQL Query:
SELECT p.category, SUM(oi.subtotal) AS total_revenue
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;

✅ Executing query...

Results (5 rows):


,category,total_revenue
0,Books,503459.98
1,Electronics,255184.65
2,Clothing,253334.02
3,Home & Garden,208793.29
4,Sports,183098.18


,category,total_revenue
0,Books,503459.98
1,Electronics,255184.65
2,Clothing,253334.02
3,Home & Garden,208793.29
4,Sports,183098.18


### Example 3: Date-based Analysis

In [22]:
# Monthly sales trend
execute_natural_language_query(
    "Show me the total sales amount for each month in 2024, ordered by month"
)


🔍 Natural Language Query: Show me the total sales amount for each month in 2024, ordered by month

⏳ Converting to SQL...

📝 Generated SQL Query:
SELECT strftime('%m', order_date) AS month, SUM(total_amount) FROM orders WHERE order_date BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY month ORDER BY month;

✅ Executing query...

Results (0 rows):


,month,SUM(total_amount)


,month,SUM(total_amount)


### Example 4: Complex Join Query

In [23]:
# Products that have never been ordered
execute_natural_language_query(
    "Find all products that have never been ordered, showing product name, category, and price"
)


🔍 Natural Language Query: Find all products that have never been ordered, showing product name, category, and price

⏳ Converting to SQL...

📝 Generated SQL Query:
SELECT product_name, category, price FROM products WHERE product_id NOT IN (SELECT product_id FROM order_items);

✅ Executing query...

Results (0 rows):


,product_name,category,price


,product_name,category,price


### Example 5: Aggregate Analysis

In [24]:
# Average order value by city
execute_natural_language_query(
    "What is the average order value for customers in each city? Show city name and average order value"
)


🔍 Natural Language Query: What is the average order value for customers in each city? Show city name and average order value

⏳ Converting to SQL...

📝 Generated SQL Query:
SELECT city, AVG(total_amount) FROM customers JOIN orders ON customers.customer_id = orders.customer_id GROUP BY city;

✅ Executing query...

Results (10 rows):


,city,AVG(total_amount)
0,Chicago,2645.957692
1,Dallas,2908.854681
2,Houston,3000.653867
3,Los Angeles,2312.001667
4,New York,3043.410536
5,Philadelphia,2544.587455
6,Phoenix,2801.892889
7,San Antonio,2542.175800
8,San Diego,3001.126071
9,San Jose,3086.102857


,city,AVG(total_amount)
0,Chicago,2645.957692
1,Dallas,2908.854681
2,Houston,3000.653867
3,Los Angeles,2312.001667
4,New York,3043.410536
5,Philadelphia,2544.587455
6,Phoenix,2801.892889
7,San Antonio,2542.175800
8,San Diego,3001.126071
9,San Jose,3086.102857


### Example 6: Status-based Analysis

In [25]:
# Order status distribution
execute_natural_language_query(
    "Count how many orders are in each status category"
)


🔍 Natural Language Query: Count how many orders are in each status category

⏳ Converting to SQL...

📝 Generated SQL Query:
SELECT status, COUNT(*) FROM orders GROUP BY status;

✅ Executing query...

Results (4 rows):


,status,COUNT(*)
0,Cancelled,128
1,Completed,121
2,Pending,117
3,Shipped,134


,status,COUNT(*)
0,Cancelled,128
1,Completed,121
2,Pending,117
3,Shipped,134


### Example 7: Your Custom Query

Now try your own analysis! Just write what you want to know in plain English.

In [26]:
# Write your own natural language query here
execute_natural_language_query(
    "Your question here..."
)


🔍 Natural Language Query: Your question here...

⏳ Converting to SQL...

📝 Generated SQL Query:
SELECT c.first_name, c.last_name, SUM(o.total_amount) AS total_spent FROM customers c JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.customer_id, c.first_name, c.last_name;

✅ Executing query...

Results (100 rows):


,first_name,last_name,total_spent
0,Ava,Johnson,21636.52
1,John,Brown,7514.17
2,James,Brown,27141.95
3,Emma,Davis,11524.59
4,Sophia,Johnson,25388.24
...,...,...,...
95,Sophia,Johnson,9055.56
96,James,Jones,9127.88
97,John,Smith,12270.86
98,Robert,Garcia,25193.57


,first_name,last_name,total_spent
0,Ava,Johnson,21636.52
1,John,Brown,7514.17
2,James,Brown,27141.95
3,Emma,Davis,11524.59
4,Sophia,Johnson,25388.24
...,...,...,...
95,Sophia,Johnson,9055.56
96,James,Jones,9127.88
97,John,Smith,12270.86
98,Robert,Garcia,25193.57



Clean Up

In [28]:
# Close database connection
conn.close()
print("Database connection closed.")

Database connection closed.
